<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [1]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

📂 Montando Google Drive...
Mounted at /content/drive
📍 Banco de Dados: /content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega/data/base-dados.db


In [2]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
🛠️ Verificando integridade das tabelas...
✅ Estrutura (Schema) validada com sucesso!

🚀 Ambiente pronto (GPU: Ativa).


In [3]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

ℹ️ O banco já contém 31062 versos. Carga inicial ignorada.


In [4]:
# Célula 4: Indexação Hierárquica e Sensores XAI (Versão Otimizada com Cache)
import spacy
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Preparação do Modelo NLP
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    import os
    os.system("python -m spacy download pt_core_news_lg")
    nlp = spacy.load("pt_core_news_lg")

def executar_indexacao_completa(db_path):
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # --- PREPARAÇÃO DO BANCO ---
        print("🧹 Limpando índices anteriores...")
        cursor.execute("DELETE FROM verso_palavra")
        cursor.execute("DELETE FROM palavra")
        cursor.execute("DELETE FROM verso_limpo")
        conn.commit()

        # Carrega palavras existentes para o cache em memória (evita query por token)
        cursor.execute("SELECT lemma, id FROM palavra")
        dicionario_palavras = dict(cursor.fetchall())

        # 2. Carga dos Versos ativos para processamento
        query = """
            SELECT v.id, v.texto
            FROM verso v
            WHERE v.processar = 'S'
        """
        df_versos = pd.read_sql_query(query, conn)
        print(f"🧠 Analisando {len(df_versos)} versos...")

        for _, row in tqdm(df_versos.iterrows(), total=len(df_versos), desc="Gramática e Sensores"):
            verso_id = row['id']
            texto = row['texto']

            if not texto or len(texto.strip()) < 3:
                continue

            doc = nlp(texto)
            total_tokens = len(doc)
            if total_tokens == 0:
                continue

            # Acumuladores de Metadados
            counts = {'ADJ': 0, 'ADV': 0, 'PROPN': 0, 'VERB': 0, 'NUM': 0, 'NOUN': 0}
            n_primeira_pessoa = 0
            palavras_sig_len = []
            is_identidade = 0
            tem_numeral = 0

            # --- PROCESSAMENTO POR TOKEN ---
            for t in doc:
                lemma_lower = t.lemma_.lower()
                pos_atual = t.pos_
                is_stop_int = 1 if t.is_stop else 0

                # Gestão de Cache da Tabela 'palavra'
                if lemma_lower not in dicionario_palavras:
                    cursor.execute("""
                        INSERT OR IGNORE INTO palavra (lemma, pos_tag, is_stop)
                        VALUES (?, ?, ?)
                    """, (lemma_lower, pos_atual, is_stop_int))

                    cursor.execute("SELECT id FROM palavra WHERE lemma = ?", (lemma_lower,))
                    resultado_id = cursor.fetchone()
                    if resultado_id:
                        dicionario_palavras[lemma_lower] = resultado_id[0]

                palavra_id = dicionario_palavras.get(lemma_lower)

                # Sensores Morfossilógicos
                if pos_atual in counts:
                    counts[pos_atual] += 1
                if t.morph.get("Person") == ["1"]:
                    n_primeira_pessoa += 1
                if pos_atual == 'NUM':
                    tem_numeral = 1

                # Sensor de Identidade (Ontologia: 'ser'/'estar' como raiz ou cópula)
                if lemma_lower in ['ser', 'estar'] and (t.dep_ in ['ROOT', 'cop']):
                    is_identidade = 1

                if not t.is_stop and not t.is_punct:
                    palavras_sig_len.append(len(t.text))

                # Índice Invertido com Hierarquia
                if palavra_id:
                    cursor.execute("""
                        INSERT INTO verso_palavra (
                            verso_id, palavra_id, posicao, head_pos, dep_relation, morph
                        ) VALUES (?, ?, ?, ?, ?, ?)
                    """, (verso_id, palavra_id, t.i, t.head.i, t.dep_, str(t.morph)))

            # --- MÉTRICAS DE NÍVEL MACRO ---
            score_emocional = (counts['ADJ'] + counts['ADV']) / total_tokens
            score_informativo = (counts['PROPN'] + counts['NUM']) / total_tokens
            score_acao = counts['VERB'] / total_tokens

            # Linha corrigida com a checagem limpa da lista
            avg_word_len = np.mean(palavras_sig_len) if palavras_sig_len else 0.0


            # Entropia Gramatical Estável
            present_tags = [v for v in counts.values() if v > 0]
            total_tags_contadas = sum(present_tags)
            if total_tags_contadas > 0:
                entropia = -sum([(v/total_tags_contadas) * np.log(v/total_tags_contadas + 1e-9) for v in present_tags])
            else:
                entropia = 0.0

            # Texto Lematizado para o Modelo Zero-Shot
            texto_limpo = " ".join([t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct])

            # Persistência dos Sensores XAI
            cursor.execute("""
                INSERT INTO verso_limpo (
                    verso_id, texto_limpo, score_emocional, score_informativo,
                    score_acao, entropia_gramatical, n_primeira_pessoa,
                    avg_word_len, is_identidade, tem_numeral
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (verso_id, texto_limpo, score_emocional, score_informativo,
                  score_acao, entropia, n_primeira_pessoa, avg_word_len,
                  is_identidade, tem_numeral))

        conn.commit()
        print("✅ Sucesso: Banco de dados atualizado e persistido.")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro de Bloqueio (Lock): {e}")
        if conn: conn.rollback()
    except Exception as e:
        print(f"❌ Erro inesperado: {e}")
        if conn: conn.rollback()
    finally:
        if conn:
            conn.close()
            print("🔒 Conexão fechada com segurança.")

# Execução automática direcionada à sua constante global
executar_indexacao_completa(DB_PATH)

🧹 Limpando índices anteriores...
🧠 Analisando 92 versos...


Gramática e Sensores:   0%|          | 0/92 [00:00<?, ?it/s]

✅ Sucesso: Banco de dados atualizado e persistido.
🔒 Conexão fechada com segurança.


In [5]:
# Célula 5: Classificação por Malha de Sementes Atômicas Compartimentadas (P&D Avançado)
import sqlite3
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados conforme o Novo Modelo Atômico
conn = sqlite3.connect(DB_PATH)

# Carrega a totalidade das subsementes de forma isolada (sem aglutinação por GROUP_CONCAT)
query_sementes = """
    SELECT ed.sentenca, ed.eixo_id, e.nome as eixo_nome
    FROM eixo_descricao ed
    JOIN eixo e ON ed.eixo_id = e.id
"""
df_sementes = pd.read_sql_query(query_sementes, conn)

# Mapeamento essencial para a IA e consolidação posterior
# Dicionário mapeia a string exata da subsemente ao ID do seu eixo correspondente
semente_para_eixo_id = dict(zip(df_sementes['sentenca'], df_sementes['eixo_id']))
labels_ia = df_sementes['sentenca'].tolist()

# Query integrada de entrada: Versos + Metadados XAI + Gênero Literário
query_input = """
    SELECT
        vl.*,
        gl.nome as genero_nome
    FROM verso_limpo vl
    JOIN verso v ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario gl ON l.genero_id = gl.id
    WHERE v.processar = 'S'
"""
df_input = pd.read_sql_query(query_input, conn)

# 2. Inicialização do Modelo BART (Zero-Shot Classification)
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0)

def classificar_malha_atomica(row, labels_ia, semente_para_eixo_id):
    # Eixo Narrativo Padrão é ID 3
    id_narrativo = 3

    # --- 1. SALVAGUARDA PARA VERSOS CURTOS (Bypass de Transição Dialética) ---
    palavras = str(row['texto_limpo']).split()
    verbos_elocucao = ['dizer', 'responder', 'continuar', 'falar', 'clamar', 'acrescentar', 'perguntar', 'replicar']

    if len(palavras) <= 4 and any(v in row['texto_limpo'] for v in verbos_elocucao):
        # Retorna vetor macro puramente direcionado ao eixo 3 (Narrativo)
        probs_macro = [0.0, 0.0, 0.0, 1.0]
        return probs_macro, "Bypass: Transição Dialética"

    # --- 2. FLUXO NORMAL: INJEÇÃO DO DNA SINTÁTICO (XAI) ---
    dna = []
    if row['n_primeira_pessoa'] > 0: dna.append("Relato Pessoal/Subjetivo")
    if row['is_identidade'] == 1: dna.append("Definição de Identidade/Estado")
    if row['tem_numeral'] == 1: dna.append("Dados Quantitativos/Inventário")

    prefixo = f"[Contexto: {', '.join(dna)}] " if dna else ""
    texto_para_ia = prefixo + row['texto_limpo']

    # Inferência multi-vetorial contra as 16 subsementes atômicas
    res = classifier(texto_para_ia, labels_ia, multi_label=False)

    # --- 3. CONSOLIDAÇÃO DE MALHA: MAPEAMENTO SUBSEMENTE -> EIXO MACRO ---
    # Inicializa acumuladores para os 4 eixos macro (0, 1, 2, 3)
    scores_macro = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
    contagem_sementes = {0: 0, 1: 0, 2: 0, 3: 0}

    # Agrega a pontuação de cada subsemente tirando a média de afinidade por eixo
    for label, score in zip(res['labels'], res['scores']):
        eixo_id = semente_para_eixo_id[label]
        scores_macro[eixo_id] += score
        contagem_sementes[eixo_id] += 1

    for eixo_id in scores_macro:
        if contagem_sementes[eixo_id] > 0:
            scores_macro[eixo_id] /= contagem_sementes[eixo_id]

    # --- 4. APLICAÇÃO DAS REGRAS DE SOBERANIA LITERÁRIA ---
    genero = row['genero_nome']
    if genero == 'Poético/Sapiencial':
        if row['score_informativo'] > 0.12 or row['tem_numeral'] == 1:
            scores_macro[id_narrativo] += 0.45
        else:
            scores_macro[id_narrativo] -= 0.15
    elif genero == 'Epístola':
        scores_macro[id_narrativo] -= 0.25

    # Ordenação e normalização estrita (Garante ordenação de IDs de 0 a 3)
    probs_vetor = [scores_macro[0], scores_macro[1], scores_macro[2], scores_macro[3]]
    probs_clipped = np.clip(probs_vetor, 0.001, 1.0)
    soma_completude = sum(probs_clipped)
    probs_macro = [p / soma_completude for p in probs_clipped]

    return probs_macro, "IA + DNA Sintático"

# 3. Processamento Iterativo com Métricas de Alta Resolução
results = []
print(f"🤖 Classificando {len(df_input)} versos na Malha Multi-Vetorial...")

for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Eixos Atômicos"):
    probs, status_sugerido = classificar_malha_atomica(row, labels_ia, semente_para_eixo_id)

    # Identificação do eixo vencedor
    idx_vencedor = np.argmax(probs)

    # Estatísticas de decisão baseadas nas distribuições macro consolidadas
    scores_ordenados = sorted(probs, reverse=True)
    gap = scores_ordenados[0] - scores_ordenados[1] if len(scores_ordenados) > 1 else 1.0
    entropia = -sum([p * np.log(p + 1e-9) for p in probs])

    # Consolidação do Status de Decisão Avançado
    status = status_sugerido
    if status == "IA + DNA Sintático":
        if gap > 0.45:
            status = "Alta Confiança"
        elif row['genero_nome'] == 'Poético/Sapiencial' and idx_vencedor == 3:
            status = "Narrativa de Moldura (XAI)"
        elif entropia > 1.1:
            status = "Ambiguidade Poética"

    # Preenchimento do payload estruturado para a base relacional
    results.append({
        'verso_id': int(row['verso_id']),
        'topico_id': int(idx_vencedor),
        'p_exaustao': probs[0],
        'p_transitoriedade': probs[1],
        'p_vazio': probs[2],
        'p_narrativo': probs[3],
        'similaridade_final': probs[idx_vencedor],
        'margem_dominancia': gap,
        'status_decisao': status,
        'entropia': entropia,
        'gap_confianca': gap
    })

# 4. Persistência Atômica dos Resultados Refinados
df_final = pd.DataFrame(results)
cursor = conn.cursor()

print("🧹 Expurgando índices de classificação obsoletos...")
cursor.execute("DELETE FROM verso_topico")

print("💾 Gravando novas distribuições de âncoras emocionais...")
df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("✨ Célula 5 atualizada para o Estado da Arte (Malha Atômica Concluída)!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

🤖 Classificando 92 versos...


Processando Eixos:   0%|          | 0/92 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🧹 Limpando classificações anteriores...
💾 Gravando novos índices existenciais ponderados...
✨ Célula 5 concluída com sucesso e protegida contra falsos-positivos dialéticos!


In [6]:
# Célula 6: Análise de Sentimento Contextualizada com Ponderação Contínua
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
import numpy as np
from tqdm.auto import tqdm

# 1. Inicializar o Analisador (BERTimbau-based para PT-BR)
print("🚀 Carregando modelo Transformer para Sentimento...")
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca de dados cruzados
conn = sqlite3.connect(DB_PATH)
query_cruzada = """
    SELECT
        v.id as verso_id,
        v.texto,
        vt.topico_id,
        vl.n_primeira_pessoa,
        vl.is_identidade
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
    JOIN verso_limpo vl ON v.id = vl.verso_id
    WHERE v.processar = 'S'
"""
df_input = pd.read_sql_query(query_cruzada, conn)

# Transforma metadados em dicionário indexado por id (O(1) de busca dentro do loop)
meta_lookup = df_input.set_index('verso_id').to_dict('index')

# 3. Execução da análise em lotes
print(f"📊 Analisando carga emocional de {len(df_input)} versículos...")
textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        v_id = ids_lote[idx]
        meta = meta_lookup.get(v_id)

        sc_pos = float(p.probas.get('POS', 0.0))
        sc_neg = float(p.probas.get('NEG', 0.0))
        sc_neu = float(p.probas.get('NEU', 0.0))

        # Ajuste Fino Continuo: Relatos em 1ª pessoa amplificam a variância da polaridade real
        multiplicador_voz = 1.5 if meta and meta['n_primeira_pessoa'] > 0 else 1.0

        # Sentimento base discreto
        sent_base = mapa_num.get(p.output, 0)

        # Cálculo contínuo baseado nas probabilidades (evita o problema de zerar o neutro bruto)
        sentimento_ajustado = float((sc_pos - sc_neg) * multiplicador_voz)

        sentimentos.append({
            'verso_id': int(v_id),
            'label': p.output,
            'sentimento_num': sent_base,
            'sentimento_ajustado': sentimento_ajustado,
            'score_pos': sc_pos,
            'score_neg': sc_neg,
            'score_neu': sc_neu
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    print("🧹 Limpando dados anteriores da tabela 'verso_sentimento'...")
    cursor.execute("DELETE FROM verso_sentimento")

    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados de forma contínua.")

    # 5. DIAGNÓSTICO FINAL: CRUZAMENTO EXISTENCIAL DO ANTÍDOTO
    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 AND vl.is_identidade = 1 THEN 1 ELSE 0 END) as Definicoes_Fortalecedoras,
            SUM(CASE WHEN vs.sentimento_num = -1 AND vl.n_primeira_pessoa > 0 THEN 1 ELSE 0 END) as Lamentos_Pessoais,
            ROUND(AVG(vs.sentimento_ajustado), 3) as Polaridade_Ponderada_Continua
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso_limpo vl ON vt.verso_id = vl.verso_id
        WHERE t.id != 3
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Ponderada_Continua DESC
    """, conn)

    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
    conn.rollback()
finally:
    conn.close()

🚀 Carregando modelo Transformer para Sentimento...


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

🧹 Limpando dados anteriores da tabela 'verso_sentimento'...

✅ Célula 6 concluída! Sentimentos processados de forma contínua.


,Eixo_Filosofico,Total_Versos,Definicoes_Fortalecedoras,Lamentos_Pessoais,Polaridade_Ponderada_Continua
